# Aligned Riemannian Cross-Subject Transfer

**SUPERSEDED TRANSFER FRAMEWORK.** The later source-aligned short-scale Full50 test was null. Do not rerun or tune this older alignment route; see `AGENTS.md` section 2e. Execution fails closed.

# 1. Setup

In [ ]:
raise RuntimeError('CLOSED superseded aligned-Riemann transfer: see AGENTS.md section 2e.')
import os, sys, json, glob, random, hashlib, builtins, platform, inspect
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy import signal, stats
from sklearn.model_selection import StratifiedKFold, GroupKFold, LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.covariance import OAS
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, confusion_matrix, f1_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
print(f"Python: {sys.version.split()[0]} | Platform: {platform.platform()} | cwd: {Path.cwd()}")

# 2. Configuration
## 2.1 Domain Defaults
## 2.2 CONFIG

In [ ]:
WORKING_DIR = Path.cwd().resolve().parent.parent
CONFIG = {
    # Paths / run identity
    "artifact_dir": str(WORKING_DIR/"artifacts"/"liu2024-aligned-riemann-transfer"), "source_extract_dir": str(WORKING_DIR/"liu2024_data"/"liu2024_figshare"/"sourcedata"),
    "experiment_name":"aligned_riemann_transfer_full50", "config_note":"Five-fold within-target evaluation of target-only and aligned pooled covariance transfer.",
    # Dataset / fixed views
    "subjects_to_use":None, "lv14_subject_ids":[1,3,7,9,10,11,14,15,17,29,31,32,37,41], "sfreq":500,
    "bands_hz":[[8,12],[12,20],[20,30],[8,30]], "windows_s":[[0,2],[1,3],[2,4]], "reference_mode":"average",
    # Models / evaluation
    "methods":["target_only","pooled_unaligned","euclidean_aligned","riemannian_recentered","prototype_shrinkage"],
    "cv_folds":5, "inner_folds":3, "regularization_grid":[0.01,0.1,1.0,10.0], "prototype_shrinkage_grid":[0.0,0.25,0.5,0.75,1.0],
    "seed":2026, "set_seed":True,
}

## 2.3 Artifact Creation and Logging Init

In [ ]:
def create_run_id():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    config_hash = hashlib.md5(json.dumps(CONFIG, sort_keys=True, default=str).encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"
RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=False)
LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1, encoding="utf-8", errors="replace")
def _safe_write_text(stream, text):
    try: stream.write(text)
    except UnicodeEncodeError:
        enc = getattr(stream, "encoding", None) or "utf-8"
        stream.write(text.encode(enc, errors="replace").decode(enc, errors="replace"))
def _timestamped_print(*args, **kwargs):
    sep, end = kwargs.pop("sep", " "), kwargs.pop("end", "\n")
    flush, target = kwargs.pop("flush", False), kwargs.pop("file", None)
    message = sep.join(str(a) for a in args)
    stamped = f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {message}" if message else ""
    for stream in ([target] if target is not None else [sys.stdout, _LOG_FILE_HANDLE]): _safe_write_text(stream, stamped + end)
    if flush: _LOG_FILE_HANDLE.flush()
builtins.print = _timestamped_print
config_path = ARTIFACT_DIR / "config.json"
with open(config_path, "w") as f: json.dump(CONFIG, f, indent=2)
print(f"Run ID:     {RUN_ID}")
print(f"Artifacts:  {ARTIFACT_DIR}")
print(f"Config:     {config_path}")

## 2.4 Reproducibility

In [ ]:
BASE_SEED = int(CONFIG["seed"])
def seed_everything(seed):
    os.environ["PYTHONHASHSEED"] = str(seed); random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
        torch.use_deterministic_algorithms(True, warn_only=True)
    except ImportError: pass
if CONFIG["set_seed"]: seed_everything(BASE_SEED)
print(f"Seed initialized: {BASE_SEED}")

# 3. Load and Prepare Data
## 3.1 Data Loading Helpers

In [ ]:
EEG_IDX = [i for i in range(30) if i != 17]
CH_NAMES = ["Fp1","Fp2","Fz","F3","F4","F7","F8","FCz","FC3","FC4","FT7","FT8","Cz","C3","C4","T3","T4","CP3","CP4","TP7","TP8","Pz","P3","P4","T5","T6","Oz","O1","O2"]
def find_files(root):
    files = sorted(Path(root).glob("sub-*/sub-*_eeg.mat"))
    if not files: raise FileNotFoundError(root)
    return files
def load_subject(path):
    eeg = sio.loadmat(path)["eeg"][0, 0]
    raw = np.asarray(eeg["rawdata"], float); y = np.asarray(eeg["label"]).ravel().astype(int) - 1
    marker = raw[:, 32]; onsets=[]
    for m in marker:
        idx=np.flatnonzero(m == 2); valid=idx[(idx >= 800) & (idx <= 1300)]
        onsets.append(int(valid[0]) if len(valid) else 1003)
    sid=int(path.parent.name.split("-")[1])
    return sid, raw[:, EEG_IDX], y, np.asarray(onsets)
def selected_files():
    keep = CONFIG["subjects_to_use"]
    return [p for p in find_files(CONFIG["source_extract_dir"]) if keep is None or int(p.parent.name.split("-")[1]) in set(keep)]

## 3.2 Fixed Covariance Views

In [ ]:
from scipy.linalg import fractional_matrix_power, logm, expm
from pyriemann.utils.mean import mean_riemann
from pyriemann.tangentspace import TangentSpace
def oas_cov(x): return OAS(store_precision=False,assume_centered=True).fit(x.T).covariance_
def extract_views(raw,onsets):
    out=[]
    for x,o in zip(raw,onsets):
        views=[]
        seg=x[:,o:o+2000]
        if CONFIG["reference_mode"]=="average": seg=seg-seg.mean(0,keepdims=True)
        elif CONFIG["reference_mode"]!="none": raise ValueError("reference_mode must be average or none")
        for lo,hi in CONFIG["bands_hz"]:
            sos=signal.butter(4,[lo,hi],btype="bandpass",fs=500,output="sos"); z=signal.sosfiltfilt(sos,seg,axis=-1)
            for a,b in CONFIG["windows_s"]: views.append(oas_cov(z[:,int(a*500):int(b*500)]))
        out.append(views)
    return np.asarray(out)
def congruence(covs,center):
    w=fractional_matrix_power(center,-0.5); return np.asarray([w@c@w.T for c in covs])
def recenter(covs,source_center,target_center):
    s=fractional_matrix_power(source_center,-0.5); t=fractional_matrix_power(target_center,0.5); return np.asarray([t@s@c@s.T@t.T for c in covs])
def feature_blocks(train,query):
    a=[]; b=[]
    for v in range(train.shape[1]):
        ts=TangentSpace(metric="riemann").fit(train[:,v]); a.append(ts.transform(train[:,v])); b.append(ts.transform(query[:,v]))
    return np.concatenate(a,1),np.concatenate(b,1)
def inner_select(X,y,grid):
    best=(-1,None)
    for C in grid:
        pred=np.zeros(len(y),int)
        for tr,va in StratifiedKFold(CONFIG["inner_folds"],shuffle=True,random_state=BASE_SEED).split(X,y):
            pred[va]=make_pipeline(StandardScaler(),LogisticRegression(C=C,max_iter=2000)).fit(X[tr],y[tr]).predict(X[va])
        best=max(best,(balanced_accuracy_score(y,pred),C))
    return best[1]

# 4. Model
## 4.1 Alignment and Tangent-Space Classifiers

# 5. Training
## 5.1 Five-Fold Within-Target Evaluation

In [ ]:
ALL={}
for p in selected_files():
    sid,raw,y,on=load_subject(p); ALL[sid]=(extract_views(raw,on),y)
SUBJECTS=sorted(ALL); FOLD_RESULTS=[]
for sid in SUBJECTS:
    Xt,yt=ALL[sid]
    source_ids=[s for s in SUBJECTS if s!=sid]
    Xs=np.concatenate([ALL[s][0] for s in source_ids]); ys=np.concatenate([ALL[s][1] for s in source_ids])
    for fold,(tr,te) in enumerate(StratifiedKFold(CONFIG["cv_folds"],shuffle=True,random_state=BASE_SEED).split(Xt,yt)):
        # Every target transform uses only target outer-training trials.
        for method in CONFIG["methods"]:
            train_cov,train_y,test_cov=Xt[tr],yt[tr],Xt[te]
            if method!="target_only": train_cov=np.concatenate([Xs,train_cov]); train_y=np.concatenate([ys,yt[tr]])
            if method=="euclidean_aligned":
                tc=np.mean(Xt[tr],axis=0); train_cov=np.asarray([[congruence(c[None],tc[v])[0] for v,c in enumerate(row)] for row in train_cov]); test_cov=np.asarray([[congruence(c[None],tc[v])[0] for v,c in enumerate(row)] for row in test_cov])
            elif method in {"riemannian_recentered","prototype_shrinkage"}:
                tc=np.asarray([mean_riemann(Xt[tr,v]) for v in range(Xt.shape[1])]); sc=np.asarray([mean_riemann(Xs[:,v]) for v in range(Xt.shape[1])])
                aligned=np.asarray([[recenter(c[None],sc[v],tc[v])[0] for v,c in enumerate(row)] for row in Xs])
                train_cov=np.concatenate([aligned,Xt[tr]]); train_y=np.concatenate([ys,yt[tr]])
            Ftr,Fte=feature_blocks(train_cov,test_cov); C=inner_select(Ftr,train_y,CONFIG["regularization_grid"])
            if method=="prototype_shrinkage":
                # Shrinkage is selected only from training data; alpha weights target vs population prototypes.
                alpha=CONFIG["prototype_shrinkage_grid"][0]
                best=-1
                for a in CONFIG["prototype_shrinkage_grid"]:
                    score=[]
                    candidate_weights=np.ones(len(train_y)); candidate_weights[-len(tr):]=1+a*len(Xs)/max(1,len(tr))
                    for itr,iva in StratifiedKFold(CONFIG["inner_folds"],shuffle=True,random_state=BASE_SEED).split(Ftr,train_y):
                        m=make_pipeline(StandardScaler(),LogisticRegression(C=C,max_iter=2000)); m.fit(Ftr[itr],train_y[itr],logisticregression__sample_weight=candidate_weights[itr]); score.append(balanced_accuracy_score(train_y[iva],m.predict(Ftr[iva])))
                    if np.mean(score)>best: best,alpha=np.mean(score),a
                weights=np.ones(len(train_y)); weights[-len(tr):]=1+alpha*len(Xs)/max(1,len(tr))
            else: alpha=None; weights=None
            model=make_pipeline(StandardScaler(),LogisticRegression(C=C,max_iter=2000)); model.fit(Ftr,train_y,logisticregression__sample_weight=weights)
            pred=model.predict(Fte); score=model.decision_function(Fte)
            FOLD_RESULTS.append({"subject_id":sid,"fold_id":fold,"method":method,"accuracy":accuracy_score(yt[te],pred),"balanced_accuracy":balanced_accuracy_score(yt[te],pred),"y_true":yt[te].tolist(),"y_pred":pred.tolist(),"score":score.tolist(),"selected_C":C,"selected_prototype_shrinkage":alpha,"confusion_matrix":confusion_matrix(yt[te],pred,labels=[0,1]).tolist()})
SUBJECT_METRICS=[]
for sid in SUBJECTS:
    for method in CONFIG["methods"]:
        rr=[r for r in FOLD_RESULTS if r["subject_id"]==sid and r["method"]==method]; yt=np.concatenate([r["y_true"] for r in rr]); yp=np.concatenate([r["y_pred"] for r in rr]); sc=np.concatenate([r["score"] for r in rr])
        SUBJECT_METRICS.append({"subject_id":sid,"method":method,"pooled_oof_accuracy":accuracy_score(yt,yp),"pooled_oof_balanced_accuracy":balanced_accuracy_score(yt,yp),"pooled_oof_auc":roc_auc_score(yt,sc)})
GLOBAL_METRICS={"primary":"pooled_oof_per_subject_then_subject_mean","methods":{m:{"mean_balanced_accuracy":float(np.mean([r["pooled_oof_balanced_accuracy"] for r in SUBJECT_METRICS if r["method"]==m]))} for m in CONFIG["methods"]}}

# 6. Results
## 6.1 Aggregate and Save Artifacts

In [ ]:
cv_results_path = ARTIFACT_DIR / "cv_results.json"
subject_metrics_path = ARTIFACT_DIR / "subject_metrics.json"
global_metrics_path = ARTIFACT_DIR / "global_metrics.json"
pd.DataFrame(FOLD_RESULTS).to_json(cv_results_path, orient="records", indent=2)
pd.DataFrame(SUBJECT_METRICS).to_json(subject_metrics_path, orient="records", indent=2)
with open(global_metrics_path, "w") as f: json.dump(GLOBAL_METRICS, f, indent=2)
run_metadata = {"run_id": RUN_ID, "artifact_dir": str(ARTIFACT_DIR), "experiment_name": CONFIG["experiment_name"], "config_note": CONFIG["config_note"], "subjects": [int(s) for s in SUBJECTS], "channel_names": CH_NAMES, "seed": BASE_SEED, "global_metrics": GLOBAL_METRICS, "performance_artifacts": {"cv_results": str(cv_results_path), "subject_metrics": str(subject_metrics_path), "global_metrics": str(global_metrics_path)}}
run_metadata_path = ARTIFACT_DIR / "run_metadata.json"
with open(run_metadata_path, "w") as f: json.dump(run_metadata, f, indent=2)
print(f"CV results saved to:      {cv_results_path}")
print(f"Subject metrics saved to: {subject_metrics_path}")
print(f"Global metrics saved to:  {global_metrics_path}")
print(f"Run metadata saved to:    {run_metadata_path}")
print(f"\nAll artifacts in: {ARTIFACT_DIR}")
try: _LOG_FILE_HANDLE.close()
except Exception: pass